In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


## Daten laden
Als erstes Schritt reproduzieren wir die Ergebnisse des Papers Grecequet et al. (2019), Select but diverse countries are reducing both climate vulnerability and CO₂ emissions

In [24]:
vuln = pd.read_csv("data/vulnerability.csv")
edgar = pd.read_csv("data/EDGAR_GHG_per_capita.csv").rename(columns={"EDGAR Country Code": "ISO3", "Country": "Name"})
START, END = 1995, 2015

def relative_change(df, start, end):
    out = df[["ISO3", "Name", str(start), str(end)]].copy()
    out["rel_change"] = (out[str(end)] - out[str(start)]) / out[str(start)] * 100
    return out[["ISO3", "Name", "rel_change"]]

# Relative Änderungen
v_rel = relative_change(vuln, START, END).rename(
    columns={"rel_change": "vuln_rel_pct"}
)
e_rel = relative_change(edgar, START, END).rename(
    columns={"rel_change": "emi_rel_pct"}
)

# Merge
df = v_rel.merge(e_rel, on=["ISO3", "Name"], how="inner").dropna()

# Gruppen wie im Paper (Quadranten)
def assign_group(vuln_x, emi_y):
    if vuln_x < 0 and emi_y > 0:
        return "Group 1: Vulnerability↓, CO2↑"
    if vuln_x < 0 and emi_y < 0:
        return "Group 2: Vulnerability↓, CO2↓"
    if vuln_x > 0 and emi_y < 0:
        return "Group 3: Vulnerability↑, CO2↓"
    if vuln_x > 0 and emi_y > 0:
        return "Group 4: Vulnerability↑, CO2↑"
    return "On axis (0-change)"

df["group"] = [assign_group(x, y) for x, y in zip(df["vuln_rel_pct"], df["emi_rel_pct"])]

# Interaktiver Plot
fig = px.scatter(
    df,
    x="vuln_rel_pct",
    y="emi_rel_pct",
    color="group",
    hover_name="Name",
    hover_data={
        "ISO3": True,
        "group": True,
        "emi_rel_pct": ":.2f",
        "vuln_rel_pct": ":.2f",
    },
    labels={
        "vuln_rel_pct": "Relative change in climate vulnerability (%) (1995–2015)",
        "emi_rel_pct": "Relative change in CO2 emissions per capita (%) (1995–2015)",
        "group": "Group"
    },
    title="Relative change (%) in climate vulnerability and CO2 emissions (1995–2015)"
)

# Null-Linien wie im Paper
fig.add_vline(x=0)
fig.add_hline(y=0)

# Optional: Achsenbereich etwas “paper-like” automatisch lassen (oder manuell setzen)
fig.show()

aut Plot haben wir die Länder identifiziert welche beides in Zeitraum von 1995 bis 2015 senken könnten. Wir geben eine Liste diesen Länder aus une werden nahand der Daten von 2015 bis 2023 untersuchen, ob die Länder auf Kurs geblieben sind. 
Gruppe 2 umfasst Länder, die zwischen 1995 und 2015 sowohl ihre Klimavulnerabilität als auch ihre CO₂-Emissionen pro Kopf reduziert haben.

In [25]:

# Group 2 (1995–2015) direkt aus dem Plot-DataFrame df
group2_95_15 = df[df["group"].str.startswith("Group 2")].copy()

print(f"Anzahl Group-2-Länder (1995–2015): {len(group2_95_15)}")

# minimal: sortieren nach Vulnerability-Veränderung (stärkste Verbesserung zuerst)
group2_95_15 = group2_95_15.sort_values("vuln_rel_pct")

# speichern
group2_95_15.to_csv("group2_countries_1995_2015.csv", index=False)

# ausgeben
group2_95_15[["ISO3", "Name", "vuln_rel_pct", "emi_rel_pct"]]


Anzahl Group-2-Länder (1995–2015): 48


,ISO3,Name,vuln_rel_pct,emi_rel_pct
143,UKR,Ukraine,-13.062776,-39.536433
35,CRI,Costa Rica,-12.630324,-3.432831
20,BWA,Botswana,-11.330722,-0.710875
102,NGA,Nigeria,-8.183811,-43.083070
79,LBN,Lebanon,-8.072126,-0.801694
76,KWT,Kuwait,-7.294468,-16.857198
134,TJK,Tajikistan,-7.084310,-6.479553
127,SVN,Slovenia,-7.011782,-9.709359
113,PRT,Portugal,-6.624624,-0.531409
50,FJI,Fiji,-6.596762,-10.418005


Warum das Paper 42 Länder findet, wir aber 48 (trotz gleicher Logik)
Die Originalstudie berichtet 42 Länder, die zwischen 1995 und 2015 sowohl CO₂-Emissionen als auch Klimavulnerabilität reduzieren konnten. In unserer Reproduktion erhalten wir 48 Länder. Diese Abweichung ist erwartbar, da wir EDGAR GHG-Emissionen pro Kopf (CO₂-Äquivalente, inkl. CH₄ und N₂O) statt CO₂ aus fossilen Brennstoffen verwenden. Dadurch ändern sich die Systemgrenzen der Emissionen, und Länder können je nach Dominanz nicht-CO₂-Treibhausgase anders klassifiziert werden. Zusätzlich können Datenquellen- und Versionsunterschiede sowie Grenzfälle nahe 0% zu leicht abweichenden Gruppenzuordnungen führen. Die qualitative Aussage bleibt jedoch stabil: Eine selektive, aber diverse Gruppe von Ländern erreicht simultane Verbesserungen in beiden Dimensionen.

Jetzt untersuchen wir nur diese Länder weiter.
Definition „auf Kurs geblieben“:
Vulnerability weiter gesunken von 2015 → 2023
CO₂-Emissionen nicht gestiegen (≤ 0 % Veränderung)

Die zwischen 1995 und 2015 identifizierten Group-2-Länder wurden anschliessend für den Zeitraum 2015–2023 weiterverfolgt. Ein Land gilt als „on track“, wenn sowohl die Klimavulnerabilität weiter gesenkt als auch die Treibhausgasemissionen pro Kopf nicht erhöht wurden. Länder, bei denen mindestens eine dieser Bedingungen nicht erfüllt ist, werden als „off track“ klassifiziert.

In [27]:
# --- 2015–2023 ---
v_15_23 = relative_change(vuln, 2015, 2023).rename(
    columns={"rel_change": "vuln_rel_15_23"}
)
e_15_23 = relative_change(edgar, 2015, 2023).rename(
    columns={"rel_change": "emi_rel_15_23"}
)

df_15_23 = v_15_23.merge(
    e_15_23[["ISO3", "emi_rel_15_23"]],
    on="ISO3",
    how="inner"
).dropna()

# Nur Group-2-Länder weiterverfolgen
track = group2_95_15.merge(
    df_15_23[["ISO3", "vuln_rel_15_23", "emi_rel_15_23"]],
    on="ISO3",
    how="left"
)

# Klassifikation: auf Kurs vs. nicht auf Kurs
track["status_2015_2023"] = track.apply(
    lambda r: "on track"
    if (r["vuln_rel_15_23"] < 0 and r["emi_rel_15_23"] <= 0)
    else "off track",
    axis=1
)

track = track.sort_values("status_2015_2023")

print(track["status_2015_2023"].value_counts())

track[[
    "ISO3",
    "Name",
    "vuln_rel_pct",
    "emi_rel_pct",
    "vuln_rel_15_23",
    "emi_rel_15_23",
    "status_2015_2023"
]]



status_2015_2023
off track    26
on track     22
Name: count, dtype: int64


,ISO3,Name,vuln_rel_pct,emi_rel_pct,vuln_rel_15_23,emi_rel_15_23,status_2015_2023
23,JAM,Jamaica,-4.117782,-29.700392,-2.092233,5.364891,off track
21,SVK,Slovakia,-4.420636,-20.830805,0.893011,-8.374154,off track
25,UZB,Uzbekistan,-3.796410,-10.104690,1.524511,9.808751,off track
26,LUX,Luxembourg,-3.787907,-26.204679,2.518719,-31.065459,off track
28,NOR,Norway,-2.854062,-5.898013,0.308814,-18.168138,off track
29,DEU,Germany,-2.818551,-19.037893,0.137542,-24.952393,off track
30,POL,Poland,-2.498070,-18.476424,0.256197,-7.112990,off track
20,NZL,New Zealand,-4.546728,-11.626861,0.447079,-15.485384,off track
34,ATG,Antigua and Barbuda,-2.174285,-2.961457,-1.927328,6.886478,off track
38,USA,United States,-2.012383,-19.428070,0.473628,-12.517388,off track


Visualisierung der Länder die in der Gruppe 2 sind 

In [28]:
# Zählung
status_counts = track["status_2015_2023"].value_counts().reset_index()
status_counts.columns = ["status", "count"]

# Balkendiagramm
fig = px.bar(
    status_counts,
    x="status",
    y="count",
    text="count",
    labels={
        "status": "Status (2015–2023)",
        "count": "Number of countries"
    },
    title="Are Group-2 countries still on track after 2015?"
)

fig.update_traces(textposition="outside")
fig.update_layout(yaxis=dict(dtick=2))

fig.show()

Die Analyse zeigt, dass nicht alle Länder, die zwischen 1995 und 2015 simultane Verbesserungen erzielen konnten, diesen Pfad nach 2015 fortsetzen. Während ein Teil der Länder weiterhin sowohl sinkende Vulnerabilität als auch stabile oder sinkende Emissionen aufweist („on track“), ist bei anderen Ländern ein Rückfall in mindestens einer Dimension zu beobachten.

In [29]:
fig2 = px.scatter(
    track,
    x="vuln_rel_15_23",
    y="emi_rel_15_23",
    color="status_2015_2023",
    hover_name="Name",
    labels={
        "vuln_rel_15_23": "Relative change in climate vulnerability (%) (2015–2023)",
        "emi_rel_15_23": "Relative change in GHG emissions per capita (%) (2015–2023)",
        "status_2015_2023": "Status"
    },
    title="Post-2015 performance of Group-2 countries"
)

fig2.add_vline(x=0)
fig2.add_hline(y=0)

fig2.show()

### GDP per capita – Daten und Verwendung

Für die ökonomische Einordnung der Länder wird das Bruttoinlandsprodukt pro Kopf (GDP per capita) verwendet. Die Daten stammen aus der Datei gdp_input.csv, welche länderspezifische GDP-pro-Kopf-Werte für mehrere Jahre enthält.

Zur Vergleichbarkeit wird für jedes Land ein fixes Referenzjahr (hier: 2015) gewählt. Dieser Wert wird anschliessend mit den zuvor identifizierten Ländern verknüpft (ISO3-Code) und dient zur Sortierung und Visualisierung nach wirtschaftlichem Entwicklungsstand.

Das GDP pro Kopf wird nicht zur Gruppenzuordnung verwendet, sondern ausschliesslich zur deskriptiven Analyse und zur Einordnung der Länder innerhalb der „on-track“-Gruppe.

In [30]:
# GDP laden
gdp = pd.read_csv("data/gdp_input.csv")  # anpassen falls Ordner anders heißt
# Beispiel: Spalten ["ISO3", "2015", "2016", ..., "2023"]

GDP_YEAR = "2015"  # oder "2020"/"2021", je nach dem was du verwenden willst

# Nur on-track Länder
on_track = track[track["status_2015_2023"] == "on track"].copy()

# GDP mergen
on_track = on_track.merge(
    gdp[["ISO3", GDP_YEAR]].rename(columns={GDP_YEAR: "gdp_pc"}),
    on="ISO3",
    how="left"
)

# nach GDP sortieren (absteigend)
on_track = on_track.sort_values("gdp_pc", ascending=False)

Die Grafik zeigt die 22 „on-track“-Länder, sortiert nach GDP pro Kopf in absteigender Reihenfolge. Sie verdeutlicht, dass nachhaltige Fortschritte in der gleichzeitigen Reduktion von Klimavulnerabilität und Emissionen nicht auf wohlhabende Staaten beschränkt sind, sondern über ein breites ökonomisches Spektrum hinweg auftreten.

In [31]:
fig = px.bar(
    on_track,
    x="Name",
    y="gdp_pc",
    text="gdp_pc",
    labels={
        "Name": "Country",
        "gdp_pc": f"GDP per capita ({GDP_YEAR})"
    },
    title="GDP per capita of on-track Group-2 countries (descending)"
)

fig.update_traces(
    texttemplate="%{text:.0f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_tickangle=-45,
    yaxis_title=f"GDP per capita ({GDP_YEAR})",
    xaxis_title="Country"
)

fig.show()


Zur vertieften Analyse werden Somalia und Vanuatu als Case Studies ausgewählt. Beide Länder gehören zu den einkommensschwächsten Staaten, konnten jedoch zwischen 1995 und 2015 sowohl ihre Klimavulnerabilität als auch ihre Emissionen pro Kopf senken und blieben auch nach 2015 auf Kurs. Trotz ähnlicher Ergebnisse unterscheiden sich die zugrunde liegenden Mechanismen deutlich.

In [32]:
COUNTRIES = ["Somalia", "Vanuatu"]
START_YEAR, END_YEAR = 1995, 2023
years = [str(y) for y in range(START_YEAR, END_YEAR + 1)]

def to_long_timeseries(df, value_name, countries):
    d = df[df["Name"].isin(countries)][["ISO3", "Name"] + years].copy()
    long = d.melt(id_vars=["ISO3", "Name"], var_name="year", value_name=value_name)
    long["year"] = long["year"].astype(int)
    return long

# --- Vulnerability long ---
vuln_long = to_long_timeseries(vuln, "Vulnerability", COUNTRIES)

# --- Emissions long ---
edgar_long = to_long_timeseries(edgar, "GHG_per_capita", COUNTRIES)

# --- Plot 1: Vulnerability (beide Länder) ---
fig_v = px.line(
    vuln_long,
    x="year",
    y="Vulnerability",
    color="Name",
    markers=True,
    title="Climate vulnerability (1995–2023): Somalia vs Vanuatu",
    labels={"year": "Year", "Vulnerability": "Vulnerability (0–1)", "Name": "Country"}
)
fig_v.add_vline(x=2015)  # Paris / Cut
fig_v.show()

# --- Plot 2: Emissions per capita (beide Länder) ---
fig_e = px.line(
    edgar_long,
    x="year",
    y="GHG_per_capita",
    color="Name",
    markers=True,
    title="GHG emissions per capita (1995–2023): Somalia vs Vanuatu",
    labels={"year": "Year", "GHG_per_capita": "GHG per capita", "Name": "Country"}
)
fig_e.add_vline(x=2015)
fig_e.show()

# --- Optional: je Land getrennte Plots (2x2) ---
for c in COUNTRIES:
    fig_v_c = px.line(
        vuln_long[vuln_long["Name"] == c],
        x="year", y="Vulnerability",
        markers=True,
        title=f"Climate vulnerability (1995–2023): {c}",
        labels={"year":"Year","Vulnerability":"Vulnerability (0–1)"}
    )
    fig_v_c.add_vline(x=2015)
    fig_v_c.show()

    fig_e_c = px.line(
        edgar_long[edgar_long["Name"] == c],
        x="year", y="GHG_per_capita",
        markers=True,
        title=f"GHG emissions per capita (1995–2023): {c}",
        labels={"year":"Year","GHG_per_capita":"GHG per capita"}
    )
    fig_e_c.add_vline(x=2015)
    fig_e_c.show()


    ### Kurze qualitative Stichpunktanalyse pro Land
1. Somalia – Fortschritte unter Fragilität

Emissionen pro Kopf ↓

Sehr niedriger Ausgangswert durch:
    - kaum Industrialisierung
    - sehr geringer Stromverbrauch pro Kopf
Keine emissionsintensiven Sektoren (z. B. Schwerindustrie)
Emissionsreduktion eher strukturell als politisch gesteuert

Vulnerability ↓
Verbesserungen trotz fragiler Staatlichkeit, v. a. durch:
    - internationale humanitäre Hilfe (Gesundheit, Ernährung, Wasser)
    - punktuelle Stabilisierung in Basisdiensten
Reduktion der Vulnerabilität nicht durch Klimapolitik im engeren Sinn, sondern durch Grundlagen der menschlichen Entwicklung

Einordnung
Fortschritte sind verletzlich und stark von externer Unterstützung abhängig. Dennoch empirisch relevant: simultane Verbesserung ist selbst unter extremen Bedingungen möglich.


2. Vanuatu – Anpassung unter hoher Exposition

Emissionen pro Kopf ↓
Sehr niedrige Emissionen aufgrund:
    - kleiner Bevölkerung
    - fehlender Industrie
Zunehmender Einsatz erneuerbarer Energien (v. a. Stromerzeugung)
Emissionspfad bleibt stabil trotz wachsender Entwicklungsbedarfe

Vulnerability ↓
Aktive und gezielte Anpassungsmaßnahmen, u. a.:
    - Katastrophenvorsorge
    - Frühwarnsysteme
    - Community-basierte Resilienzstrategien
Starke Rolle internationaler Klimafinanzierung und Entwicklungsprogramme

Einordnung
Vanuatu zeigt, dass Vulnerabilität reduzierbar ist, auch bei sehr hoher physischer Klimaexposition
Anpassungspolitik wirkt selbst bei begrenzten Ressourcen

| Aspekt                               | Somalia                            | Vanuatu                              |
| ------------------------------------ | ---------------------------------- | ------------------------------------ |
| Einkommensniveau                     | Sehr niedrig                       | Niedrig                              |
| Staatliche Kapazität                 | Sehr gering / fragil               | Begrenzt, aber stabil                |
| Klimaexposition                      | Mittel bis hoch                    | Sehr hoch (Inselstaat)               |
| Emissionsstruktur                    | Extrem niedrig, kaum Industrie     | Sehr niedrig, kaum Industrie         |
| Haupttreiber Emissionsrückgang       | Strukturelle Unterentwicklung      | Niedriger Ausgangswert + Erneuerbare |
| Haupttreiber Vulnerabilitätsrückgang | Humanitäre Hilfe, Basisentwicklung | Gezielte Anpassungspolitik           |
| Rolle internationaler Akteure        | Dominant                           | Wichtig, aber eingebettet            |
| Nachhaltigkeit der Fortschritte      | Fragil                             | Vergleichsweise robust               |
| Charakter des Erfolgs                | Unbeabsichtigter Nebeneffekt       | Politisch intendiert                 |


Beide Länder zeigen, dass simultane Reduktionen von Klimavulnerabilität und Emissionen auch in einkommensschwachen Kontexten möglich sind. Allerdings beruhen diese Fortschritte auf sehr unterschiedlichen Mechanismen: Somalia profitiert primär von strukturellen und humanitären Faktoren, während Vanuatu gezielt in Anpassungsmassnahmen investiert. Die Ergebnisse unterstreichen, dass „on track“ kein einheitliches Erfolgsmodell darstellt, sondern kontextspezifische Pfade widerspiegelt.
